# CrossBDA: Train Scale-MAE voi Fourier Domain Adaptation (FDA)

Muc tieu: Huan luyen full 50 epochs mo hinh Scale-MAE voi FDA Augmentation tren Kaggle GPU.

In [ ]:
# Buoc 1: Cai dat thu vien va clone repo
!pip install -q segmentation-models-pytorch timm albumentations shapely pyyaml

import os, sys
work_dir = '/kaggle/working/CrossBDA'
if not os.path.exists(work_dir):
    print('Cloning repository CrossBDA...')
    !git clone https://github.com/mhieudzvcl/CrossBDA.git {work_dir}

%cd {work_dir}
sys.path.insert(0, work_dir)
print('Working directory:', os.getcwd())

In [ ]:
# Buoc 2: Kiem tra GPU va Tu dong ghep cap du lieu xBD & Ida-BD
import torch, glob, os
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

print('Dang quet toan bo file trong /kaggle/input...')
all_files = glob.glob('/kaggle/input/**', recursive=True)

pre_map = {}
post_map = {}
lbl_map = {}

for f in all_files:
    if os.path.isdir(f):
        continue
    fname = os.path.basename(f)
    if 'ida' in f.lower():
        continue
    if fname.endswith('_pre_disaster.png'):
        base = fname.replace('_pre_disaster.png', '')
        pre_map[base] = f
    elif fname.endswith('_post_disaster.png'):
        base = fname.replace('_post_disaster.png', '')
        post_map[base] = f
    elif fname.endswith('_post_disaster.json'):
        base = fname.replace('_post_disaster.json', '')
        lbl_map[base] = f
    elif ('target' in fname.lower() or 'mask' in fname.lower()) and fname.endswith('.png'):
        base = fname.replace('_post_disaster_target.png', '').replace('_target.png', '').replace('_mask.png', '')
        if base not in lbl_map:
            lbl_map[base] = f

print(f'Tim thay: {len(pre_map)} anh pre, {len(post_map)} anh post, {len(lbl_map)} file nhan (JSON/PNG)')

common_bases = sorted(list(set(pre_map.keys()) & set(post_map.keys()) & set(lbl_map.keys())))
print(f'==> So mau ghep cap thanh cong: {len(common_bases)} mau!')

if len(common_bases) == 0:
    print('CANH BAO: Khong ghep duoc cap nao! Kiem tra danh sach thu muc /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        if files:
            print(f'  Folder: {root} -> Files ({len(files)}): {files[:3]}')
    raise ValueError('Khong tim thay mau hop le trong xd-dataset!')

ida_images = [f for f in all_files if 'ida' in f.lower() and f.endswith('.png') and not os.path.isdir(f)]
print(f'==> So anh bao Ida tim thay: {len(ida_images)} anh!')
assert len(ida_images) > 0, 'Khong tim thay anh bao Ida nao trong /kaggle/input!'

In [ ]:
# Buoc 3: Dinh nghia FDA Transform va Lop Dataset Tu Do
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from src.dataset import json_to_mask, get_train_transforms, get_val_transforms, IMG_SIZE

def fda_transform_numpy(src_img, tgt_img, beta=0.01):
    h, w, c = src_img.shape
    tgt_img_resized = np.array(Image.fromarray(tgt_img.astype(np.uint8)).resize((w, h), Image.BILINEAR), dtype=np.float32)
    
    src_fda = np.zeros_like(src_img, dtype=np.float32)
    for ch in range(c):
        src_fft = np.fft.fft2(src_img[:, :, ch])
        tgt_fft = np.fft.fft2(tgt_img_resized[:, :, ch])
        
        fft_src_shift = np.fft.fftshift(src_fft)
        tgt_fft_shift = np.fft.fftshift(tgt_fft)
        
        amp_src, pha_src = np.abs(fft_src_shift), np.angle(fft_src_shift)
        amp_tgt = np.abs(tgt_fft_shift)
        
        b_h, b_w = int(h * beta), int(w * beta)
        c_h, c_w = h // 2, w // 2
        amp_src[c_h - b_h : c_h + b_h, c_w - b_w : c_w + b_w] = amp_tgt[c_h - b_h : c_h + b_h, c_w - b_w : c_w + b_w]
        
        fft_swapped = np.fft.ifftshift(amp_src * np.exp(1j * pha_src))
        src_fda[:, :, ch] = np.real(np.fft.ifft2(fft_swapped))
        
    return np.clip(src_fda, 0, 255).astype(np.uint8)

class PairedXBDDataset(Dataset):
    def __init__(self, bases, pre_map, post_map, lbl_map, transform=None, ida_list=None, fda_beta=0.01, fda_p=0.5):
        self.bases = bases
        self.pre_map = pre_map
        self.post_map = post_map
        self.lbl_map = lbl_map
        self.transform = transform
        self.ida_list = ida_list if ida_list else []
        self.fda_beta = fda_beta
        self.fda_p = fda_p

    def __len__(self):
        return len(self.bases)

    def _apply_fda(self, img_np):
        if len(self.ida_list) == 0 or np.random.random() > self.fda_p:
            return img_np
        tgt_path = np.random.choice(self.ida_list)
        tgt_img  = np.array(Image.open(tgt_path).convert('RGB'), dtype=np.float32)
        return fda_transform_numpy(img_np, tgt_img, beta=self.fda_beta)

    def __getitem__(self, idx):
        base = self.bases[idx]
        pre_img  = np.array(Image.open(self.pre_map[base]).convert('RGB'), dtype=np.uint8)
        post_img = np.array(Image.open(self.post_map[base]).convert('RGB'), dtype=np.uint8)
        
        lbl_path = self.lbl_map[base]
        if lbl_path.endswith('.json'):
            dmg_mask = json_to_mask(lbl_path)
        else:
            dmg_mask = np.array(Image.open(lbl_path), dtype=np.uint8)
        loc_mask = (dmg_mask > 0).astype(np.uint8)

        if len(self.ida_list) > 0:
            pre_img  = self._apply_fda(pre_img)
            post_img = self._apply_fda(post_img)

        if self.transform:
            augmented = self.transform(
                image=pre_img, image2=post_img,
                mask=loc_mask, mask2=dmg_mask,
            )
            pre_img  = augmented['image']
            post_img = augmented['image2']
            loc_mask = augmented['mask'].long()
            dmg_mask = augmented['mask2'].long()
        else:
            pre_img  = torch.from_numpy(pre_img.transpose(2, 0, 1)).float() / 255.0
            post_img = torch.from_numpy(post_img.transpose(2, 0, 1)).float() / 255.0
            loc_mask = torch.from_numpy(loc_mask).long()
            dmg_mask = torch.from_numpy(dmg_mask).long()

        return {
            'pre_img': pre_img,
            'post_img': post_img,
            'loc_mask': loc_mask,
            'dmg_mask': dmg_mask,
            'name': base
        }

print('Lop PairedXBDDataset da san sang!')

In [ ]:
# Buoc 4: Huan luyen Scale-MAE + FDA (Full 50 Epochs)
import yaml
from pathlib import Path
from tqdm import tqdm
from src.models.factory import create_model
from src.losses import CombinedLoss
from src.metrics import MetricAccumulator, AverageMeter

EPOCHS        = 50
BATCH_SIZE    = 4
LEARNING_RATE = 3e-5
FDA_BETA      = 0.01
CHECKPOINT_TO_RESUME = ''

cfg = {
    'seed': 42,
    'model': {'encoder': 'scalemae', 'encoder_weights': 'imagenet'},
    'loss': {'w_loc': 0.4, 'w_dmg': 0.6}
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Khoi tao mo hinh Scale-MAE...')
model = create_model(cfg).to(device)

if CHECKPOINT_TO_RESUME and os.path.exists(CHECKPOINT_TO_RESUME):
    print('Nhan trong so tu checkpoint:', CHECKPOINT_TO_RESUME)
    ckpt = torch.load(CHECKPOINT_TO_RESUME, map_location='cpu')
    state = {k.replace('module.', ''): v for k, v in (ckpt.get('model_state', ckpt)).items()}
    model.load_state_dict(state, strict=False)
    print('Da load checkpoint thanh cong.')

if torch.cuda.device_count() > 1:
    print(f'Su dung {torch.cuda.device_count()} GPUs qua DataParallel')
    model = torch.nn.DataParallel(model)

# Chia tap Train / Val ngau nhien theo seed
n_total = len(common_bases)
n_val = int(n_total * 0.15)
n_train = n_total - n_val

random.seed(42)
shuffled_bases = common_bases.copy()
random.shuffle(shuffled_bases)
train_bases, val_bases = shuffled_bases[:n_train], shuffled_bases[n_train:]

train_ds = PairedXBDDataset(
    train_bases, pre_map, post_map, lbl_map,
    transform=get_train_transforms(512),
    ida_list=ida_images, fda_beta=FDA_BETA, fda_p=0.5
)
val_ds = PairedXBDDataset(
    val_bases, pre_map, post_map, lbl_map,
    transform=get_val_transforms(512),
    ida_list=[]
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'So mau Train: {len(train_ds)} | So mau Val: {len(val_ds)}')

criterion = CombinedLoss(w_loc=0.4, w_dmg=0.6)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_score = 0.0
save_path = '/kaggle/working/best_model_ScaleMAE_FDA.pth'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_meter = AverageMeter()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Train]', leave=False)
    for batch in pbar:
        pre    = batch['pre_img'].to(device, non_blocking=True)
        post   = batch['post_img'].to(device, non_blocking=True)
        loc_gt = batch['loc_mask'].to(device, non_blocking=True)
        dmg_gt = batch['dmg_mask'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loc_logits, dmg_logits = model(pre, post)
            loss, _ = criterion(loc_logits, dmg_logits, loc_gt, dmg_gt)
            
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        train_loss_meter.update(loss.item())
        pbar.set_postfix(loss=f'{train_loss_meter.avg:.4f}')
        
    scheduler.step()
    
    model.eval()
    val_loss_meter = AverageMeter()
    accumulator    = MetricAccumulator()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Val]', leave=False):
            pre    = batch['pre_img'].to(device, non_blocking=True)
            post   = batch['post_img'].to(device, non_blocking=True)
            loc_gt = batch['loc_mask'].to(device, non_blocking=True)
            dmg_gt = batch['dmg_mask'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                loc_logits, dmg_logits = model(pre, post)
                loss, _ = criterion(loc_logits, dmg_logits, loc_gt, dmg_gt)
            val_loss_meter.update(loss.item())
            accumulator.update(loc_logits, dmg_logits, loc_gt, dmg_gt)
            
    metrics = accumulator.compute()
    score   = metrics['xview2_score']
    print(f'Epoch {epoch:02d} | Train: {train_loss_meter.avg:.4f} | Val: {val_loss_meter.avg:.4f} | Loc: {metrics["f1_loc"]:.4f} | Dmg: {metrics["f1_dmg_macro"]:.4f} | Score: {score:.4f}')
    
    if score > best_score:
        best_score = score
        model_to_save = model.module if hasattr(model, 'module') else model
        ckpt = {
            'epoch': epoch,
            'model_state': model_to_save.state_dict(),
            'score': score,
            'metrics': metrics,
            'config': cfg
        }
        torch.save(ckpt, save_path)
        print(f'  New best score: {best_score:.4f} -> Saved to: {save_path}')

print(f'Training hoan tat. Best Score: {best_score:.4f}')

In [ ]:
# Buoc 5: Kiem tra file checkpoint da luu
import os
save_path = '/kaggle/working/best_model_ScaleMAE_FDA.pth'
if os.path.exists(save_path):
    size_mb = os.path.getsize(save_path) / (1024 * 1024)
    print(f'Check file thanh cong: {save_path} ({size_mb:.2f} MB)')
    print('Ban co the tai file ve may tai tab Output ben phai.')
else:
    print('Chua tim thay file checkpoint.')